In [2]:
import pandas as pd
import numpy as np

In [3]:
# read csv files
expert_r = pd.read_csv("expert_r_extract.csv")
user_r = pd.read_csv("user_r_extract.csv")
movie_lookup = pd.read_csv("movie_lookup.csv")

C:\Users\rolan\AppData\Local\Temp\ipykernel_36648\158015450.py:3: DtypeWarning: Columns (0: idvscore) have mixed types. Specify dtype option on import or set low_memory=False.
  user_r = pd.read_csv("user_r_extract.csv")


In [4]:
# checking type(s) after warning
user_r['idvscore'].apply(type).value_counts()

idvscore
<class 'str'>      186288
<class 'float'>    133374
Name: count, dtype: int64

In [5]:
print(user_r[user_r['idvscore'].apply(lambda x: isinstance(x, str))]['idvscore'].unique())

['8.0' '9.0' '10.0' '6.0' '7.0' '4.0' '5.0' '3.0' '0.0' '2.0' '1.0' '7'
 '9' '2' '10' '8' '0' '5' '3' '4' '6' '1' ':;;$$'
 ' ayfkm!! wtf!!) Ten reasons I respectfully disagree: (10) Really? Every character in the movie is a caricature. And while I liked me my Beavis & Butt-Head (huh-huh rules) for its then-revelatory reductionist critique of the then-dying music video industr'
 ' cargo carriers'
 'few i think?? i cam out the theather seeing people smile and hearing people say that was worth the waith no bull! I can t wait for a sequel it will be better than this one or just as good as the this one???it just a great winning film in your mind that u'
 ';!?']


In [6]:
broken_mask = user_r['idvscore'].astype(str).str.len() > 5  # scores are short; junk text is long
broken_rows = user_r[broken_mask]
print(len(broken_rows))
broken_rows

3


,url,reviewer,idvscore,source
59309,https://www.metacritic.com/movie/extract,NaN,ayfkm!! wtf!!) Ten reasons I respectfully dis...,user
124298,https://www.metacritic.com/movie/rogue-one-a-s...,storm-troopers,cargo carriers,user
270424,https://www.metacritic.com/movie/underworld-ev...,NaN,few i think?? i cam out the theather seeing pe...,user


In [7]:
# keep only rows where idvscore is short enough to be a real score, not review text
user_r = user_r[user_r['idvscore'].astype(str).str.len() <= 5].copy()

# now safe to convert to numeric
user_r['idvscore'] = pd.to_numeric(user_r['idvscore'], errors='coerce')

print(user_r.shape)
print(user_r['idvscore'].isna().sum())  # should be low/zero now

(316255, 4)
2


In [8]:
# what got removed?
removed = 319662 - 316255
print(removed)

# were there any 4-5 character values that were actually valid and got kept correctly?
# (just confirming nothing weird is left)
print(user_r['idvscore'].describe())

3407
count    316253.000000
mean          6.608184
std           3.185721
min           0.000000
25%           5.000000
50%           7.000000
75%           9.000000
max          10.000000
Name: idvscore, dtype: float64


In [9]:
# dropping the two broken values
user_r = user_r.dropna(subset=['idvscore'])
print(user_r.shape)

(316253, 4)


In [10]:
# checking xpert_r for the same issue
expert_r['idvscore'].apply(type).value_counts()

idvscore
<class 'float'>    238973
Name: count, dtype: int64

In [11]:
# converting idvscore into real numbers and deleting ant row where that wasnt possible
expert_r['idvscore'] = pd.to_numeric(expert_r['idvscore'], errors='coerce')
expert_r = expert_r.dropna(subset=['idvscore'])

In [12]:
# Cleaning up reviewer and rev
reviews = pd.concat([expert_r, user_r], ignore_index=True)
print(reviews.shape)   # expect (555226, 4) roughly

# clean stray quotes and whitespace off reviewer
reviews['reviewer'] = reviews['reviewer'].astype(str).str.strip().str.strip("'\"")
reviews['reviewer'] = reviews['reviewer'].replace('None', pd.NA)

reviews[['reviewer']].head(10)

(555224, 4)


,reviewer
0,Andrew O'Hehir
1,A.O. Scott
2,NaN
3,Noel Murray
4,Joshua Rothkopf
5,NaN
6,Bill Goodykoontz
7,Kimberley Jones
8,Peter Travers
9,V.A. Musetto


In [13]:
# where are the text reviews
print(expert_r.columns.tolist())
print(user_r.columns.tolist())

['url', 'reviewer', 'idvscore', 'source']
['url', 'reviewer', 'idvscore', 'source']


In [ ]:
# found out the text is still in expert_r.csv
expert_r_raw = pd.read_csv("expert_r.csv")
print(expert_r_raw.columns.tolist())
expert_r_raw.head(3)

['url', 'idvscore', 'reviewer', 'dateP', 'Rev', 'WC', 'Analytic', 'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function', 'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron', 'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj', 'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad', 'social', 'family', 'friend', 'female', 'male', 'cogproc', 'insight', 'cause', 'discrep', 'tentat', 'certain', 'differ', 'percept', 'see', 'hear', 'feel', 'bio', 'body', 'health', 'sexual', 'ingest', 'drives', 'affiliation', 'achieve', 'power', 'reward', 'risk', 'focuspast', 'focuspresent', 'focusfuture', 'relativ', 'motion', 'space', 'time', 'work', 'leisure', 'home', 'money', 'relig', 'death', 'informal', 'swear', 'netspeak', 'assent', 'nonflu', 'filler', 'AllPunc', 'Period', 'Comma', 'Colon', 'SemiC', 'QMark', 'Exclam', 'Dash', 'Quote', 'Apostro', 'Parenth', 'OtherP']


,url,idvscore,reviewer,dateP,Rev,WC,Analytic,Clout,Authentic,Tone,...,Comma,Colon,SemiC,QMark,Exclam,Dash,Quote,Apostro,Parenth,OtherP
0,https://www.metacritic.com/movie/bronson,100.0,"""Andrew O'Hehir""",None,'Bronson owes a little or a lot to Kubrick s ...,25,73.88,11.52,6.70,25.77,...,4.00,0.0,0.0,0.0,0.0,0.0,0,8.00,0.0,0.0
1,https://www.metacritic.com/movie/bronson,90.0,'A.O. Scott',None,'Bronson invites you to admire its protagonis...,30,13.07,97.69,1.00,99.00,...,6.67,0.0,0.0,0.0,0.0,0.0,0,6.67,0.0,0.0
2,https://www.metacritic.com/movie/bronson,90.0,None,None,'Whether it s Peterson/Bronson s more theatri...,40,72.69,77.33,7.84,2.89,...,7.50,0.0,0.0,0.0,0.0,12.5,0,5.00,0.0,2.5


In [ ]:
# and in here
user_r_raw = pd.read_csv("user_r.csv")
print(user_r_raw.columns.tolist())

C:\Users\rolan\AppData\Local\Temp\ipykernel_36648\3523399708.py:1: DtypeWarning: Columns (0: idvscore, 1: thumbsUp, 2: thumbsTot) have mixed types. Specify dtype option on import or set low_memory=False.
  user_r_raw = pd.read_csv("user_r.csv")


['url', 'idvscore', 'reviewer', 'dateP', 'Rev', 'thumbsUp', 'thumbsTot', 'WC', 'Analytic', 'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function', 'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron', 'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj', 'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo', 'anx', 'anger', 'sad', 'social', 'family', 'friend', 'female', 'male', 'cogproc', 'insight', 'cause', 'discrep', 'tentat', 'certain', 'differ', 'percept', 'see', 'hear', 'feel', 'bio', 'body', 'health', 'sexual', 'ingest', 'drives', 'affiliation', 'achieve', 'power', 'reward', 'risk', 'focuspast', 'focuspresent', 'focusfuture', 'relativ', 'motion', 'space', 'time', 'work', 'leisure', 'home', 'money', 'relig', 'death', 'informal', 'swear', 'netspeak', 'assent', 'nonflu', 'filler', 'AllPunc', 'Period', 'Comma', 'Colon', 'SemiC', 'QMark', 'Exclam', 'Dash', 'Quote', 'Apostro', 'Parenth', 'OtherP']


In [18]:
# check for mixed type issue
print(expert_r_raw['idvscore'].apply(type).value_counts())
print(user_r_raw['idvscore'].apply(type).value_counts())

idvscore
<class 'float'>    238973
Name: count, dtype: int64
idvscore
<class 'float'>    279157
<class 'str'>       40331
<class 'int'>         174
Name: count, dtype: int64


In [19]:
# check ints
print(user_r_raw[user_r_raw['idvscore'].apply(lambda x: isinstance(x, int))]['idvscore'].unique())

[10 8 9 7 6 2]


In [20]:
# checking str values
print(user_r_raw[user_r_raw['idvscore'].apply(lambda x: isinstance(x, str))]['idvscore'].unique())

['7' '9' '2' '10' '8' '0' '5' '3' '4' '6' '1' ':;;$$'
 ' ayfkm!! wtf!!) Ten reasons I respectfully disagree: (10) Really? Every character in the movie is a caricature. And while I liked me my Beavis & Butt-Head (huh-huh rules) for its then-revelatory reductionist critique of the then-dying music video industr'
 ' cargo carriers'
 'few i think?? i cam out the theather seeing people smile and hearing people say that was worth the waith no bull! I can t wait for a sequel it will be better than this one or just as good as the this one???it just a great winning film in your mind that u'
 ';!?']


In [21]:
# keep only rows where idvscore is short enough to be a real score, not review text
user_r_raw = user_r_raw[user_r_raw['idvscore'].astype(str).str.len() <= 5].copy()

# convert everything to real numbers
user_r_raw['idvscore'] = pd.to_numeric(user_r_raw['idvscore'], errors='coerce')
user_r_raw = user_r_raw.dropna(subset=['idvscore'])

expert_r_raw['idvscore'] = pd.to_numeric(expert_r_raw['idvscore'], errors='coerce')
expert_r_raw = expert_r_raw.dropna(subset=['idvscore'])

print(user_r_raw.shape)
print(expert_r_raw.shape)

(316253, 100)
(238971, 98)


In [ ]:
# adding "source" to differentiate between user and expert later
expert_r_raw['source'] = 'expert'
user_r_raw['source'] = 'user'

reviews = pd.concat([expert_r_raw, user_r_raw], ignore_index=True) # concat stacks them together
print(reviews.shape)

(555224, 101)


In [23]:
# keep only the columns we actually need
reviews = reviews[['url', 'reviewer', 'Rev', 'idvscore', 'source']]

# fixing encoding
def fix_encoding(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

# clean reviewer: strip stray quotes/whitespace, fix "None" strings, fix encoding
reviews['reviewer'] = reviews['reviewer'].astype(str).str.strip().str.strip("'\"")
reviews['reviewer'] = reviews['reviewer'].replace('None', pd.NA)
reviews['reviewer'] = reviews['reviewer'].apply(lambda x: fix_encoding(x) if pd.notna(x) else x)

# clean Rev the same way
reviews['Rev'] = reviews['Rev'].astype(str).str.strip().str.strip("'\"")
reviews['Rev'] = reviews['Rev'].replace('None', pd.NA)
reviews['Rev'] = reviews['Rev'].apply(lambda x: fix_encoding(x) if pd.notna(x) else x)

print(reviews.shape)
reviews.head(10)

(555224, 5)


,url,reviewer,Rev,idvscore,source
0,https://www.metacritic.com/movie/bronson,Andrew O'Hehir,Bronson owes a little or a lot to Kubrick s Cl...,100.0,expert
1,https://www.metacritic.com/movie/bronson,A.O. Scott,Bronson invites you to admire its protagonist ...,90.0,expert
2,https://www.metacritic.com/movie/bronson,NaN,Whether it s Peterson/Bronson s more theatrica...,90.0,expert
3,https://www.metacritic.com/movie/bronson,Noel Murray,There are two Bronsons on display here: the im...,83.0,expert
4,https://www.metacritic.com/movie/bronson,Joshua Rothkopf,Refn has somehow found his way to an authentic...,80.0,expert
5,https://www.metacritic.com/movie/bronson,NaN,"He s neither victim nor hero, but a man who, i...",80.0,expert
6,https://www.metacritic.com/movie/bronson,Bill Goodykoontz,"This is unhinged genius, an amazing piece of a...",80.0,expert
7,https://www.metacritic.com/movie/bronson,Kimberley Jones,Refn s artful and energetic film never goes fu...,78.0,expert
8,https://www.metacritic.com/movie/bronson,Peter Travers,This movie and Hardy s electrifying performanc...,75.0,expert
9,https://www.metacritic.com/movie/bronson,V.A. Musetto,Tom Hardy gives an amazing performance as Pete...,75.0,expert


In [ ]:
# Review text sourced from expert_r.csv/user_r.csv has some apostrophes missing (e.g. 'Kubrick's' → 'Kubrick s'), likely from the original data collection/export process — noted as a known limitation, doesn't affect movieid/reviewer/score accuracy

" 'Bronson owes a little or a lot to Kubrick s Clockwork Orange, but if that s a crime I wish more people would commit it.'"


In [26]:
print(movie_lookup.columns.tolist())
movie_lookup.head(3)

['movieid', 'title', 'awards', 'cast', 'director', 'genre', 'metascore', 'userscore']


,movieid,title,awards,cast,director,genre,metascore,userscore
0,1,Fantasia 2000,#63MostDiscussedMovieof2000,"James Levine,Steve Martin",Don Hahn,"Fantasy,Music,Animation,Family",59,7.2
1,2,Lupin III: The Castle of Cagliostro,#77BestMovieof2000,"Eiko Masuyama,GorÃ´ Naya,IchirÃ´ Nagai,Kiyoshi...",Hayao Miyazaki,"Adventure,Fantasy,Animation,Family",72,8.1
2,3,My Dog Skip,NaN,"Diane Lane,Frankie Muniz,Kevin Bacon",Jay Russell,"Drama,Sport,Family",61,7.9


In [27]:
# use meta.csv to bridge 'reviews' & 'movies' (match url and title)

meta = pd.read_csv("meta.csv", encoding="utf-8")

# step 1: get title from url via meta.csv
reviews = reviews.merge(meta[['url', 'title']], on='url', how='left')
print(reviews['title'].isna().sum())   # how many urls didn't match meta.csv

# step 2: get movieid from title via movie_lookup.csv
reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # how many titles didn't match movie_lookup

0
37


In [28]:
# check 37 titles
unmatched_titles = reviews[reviews['movieid'].isna()]['title'].unique()
print(len(unmatched_titles))
print(unmatched_titles)

2
<ArrowStringArray>
['Cet amour-lÃ ', 'Caterina va in cittÃ ']
Length: 2, dtype: str


In [31]:
# correct broken titles
reviews['title'] = reviews['title'].replace({
    'Cet amour-lÃ ': 'Cet amour-là',
    'Caterina va in cittÃ ': 'Caterina va in città'
})

reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0

2913


In [32]:
# reloading meta.vsc and rebuilding reviews from scratch

# reload meta completely fresh, undoing the earlier blanket fix_encoding on title
meta = pd.read_csv("meta.csv", encoding="utf-8")

# rebuild reviews from the raw dataframes (these should still be clean/untouched)
reviews = pd.concat([expert_r_raw, user_r_raw], ignore_index=True)
reviews = reviews[['url', 'reviewer', 'Rev', 'idvscore', 'source']]

reviews['reviewer'] = reviews['reviewer'].astype(str).str.strip().str.strip("'\"")
reviews['reviewer'] = reviews['reviewer'].replace('None', pd.NA)
reviews['reviewer'] = reviews['reviewer'].apply(lambda x: fix_encoding(x) if pd.notna(x) else x)
reviews['Rev'] = reviews['Rev'].astype(str).str.strip().str.strip("'\"")
reviews['Rev'] = reviews['Rev'].replace('None', pd.NA)

# merge with FRESH meta (title not touched by blanket fix)
reviews = reviews.merge(meta[['url', 'title']], on='url', how='left')

# targeted fix on just the 2 known-broken titles
reviews['title'] = reviews['title'].replace({
    'Cet amour-lÃ ': 'Cet amour-là',
    'Caterina va in cittÃ ': 'Caterina va in città'
})

reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0

37


In [34]:
# locate complete broken titles
raw_title = reviews[reviews['title'].str.contains('amour', na=False)]['title'].iloc[0]
print(repr(raw_title))

raw_title2 = reviews[reviews['title'].str.contains('Caterina', na=False)]['title'].iloc[0]
print(repr(raw_title2))

'Cet amour-lÃ\xa0'
'Caterina va in cittÃ\xa0'


In [39]:
# fixing broken titles
reviews['title'] = reviews['title'].replace({
    'Cet amour-lÃ\xa0': 'Cet amour-là',
    'Caterina va in cittÃ\xa0': 'Caterina va in città'
})
check = reviews[reviews['title'].str.contains('amour|Caterina', na=False)]['title'].unique()
print(check)


<ArrowStringArray>
['Cet amour-là', 'L'amour fou', 'Caterina va in città']
Length: 3, dtype: str


In [37]:
print(reviews.columns.tolist())
print(movie_lookup.columns.tolist())

['url', 'reviewer', 'Rev', 'idvscore', 'source', 'title', 'movieid_x', 'movieid_y']
['movieid', 'title', 'awards', 'cast', 'director', 'genre', 'metascore', 'userscore']


In [40]:
# cleaning up leftover duplicate columns and run merge
reviews = reviews.drop(columns=['movieid_x', 'movieid_y'], errors='ignore')
reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0

37


In [41]:
# finding unmatched titles again
unmatched_titles = reviews[reviews['movieid'].isna()]['title'].unique()
print(len(unmatched_titles))
print(unmatched_titles)

2
<ArrowStringArray>
['Cet amour-là', 'Caterina va in città']
Length: 2, dtype: str


In [42]:
# check movie lookup csv for same issue with same titles
check2 = movie_lookup[movie_lookup['title'].str.contains('amour|Caterina', na=False)]['title']
print(check2.apply(repr))

1261            'Cet amour-lÃ'
2263    'Caterina va in cittÃ'
5026             "L'amour fou"
Name: title, dtype: str


In [43]:
# fixing it the same way as before

movie_lookup['title'] = movie_lookup['title'].replace({
    'Cet amour-lÃ': 'Cet amour-là',
    'Caterina va in cittÃ': 'Caterina va in città'
})

reviews = reviews.drop(columns=['movieid_x', 'movieid_y', 'movieid'], errors='ignore')
reviews = reviews.merge(movie_lookup[['movieid', 'title']], on='title', how='left')
print(reviews['movieid'].isna().sum())   # should be 0 now

0


In [44]:
# finishing up, according to ERD columns

reviews_final = reviews[['url', 'movieid', 'reviewer', 'Rev', 'idvscore', 'source']].copy()

# rename to match SQL naming convention
reviews_final = reviews_final.rename(columns={
    'reviewer': 'reviewer',
    'Rev': 'review',
    'idvscore': 'idvscore'
})

# generate a review_id, sorted by movieid
reviews_final = reviews_final.sort_values('movieid').reset_index(drop=True)
reviews_final.insert(0, 'reviewid', reviews_final.index + 1)

# dropping the url helper column now that movieid is resolved
reviews_final = reviews_final.drop(columns=['url'])

print(reviews_final.shape)
reviews_final.head(10)

(889874, 6)


,reviewid,movieid,reviewer,review,idvscore,source
0,1,1,Jessica Winter,Arriving just after the best year for animated...,40.0,expert
1,2,1,Lisa Alspector,The rest of these animated sequences...depend ...,40.0,expert
2,3,1,Gemma Files,"Expertly done, and a real joy to watch.",70.0,expert
3,4,1,Maitland McDonagh,"Like the original Fantasia s eight segments, t...",70.0,expert
4,5,1,Kenneth Turan,"Grainy as it looks in its massive Imax blowup,...",60.0,expert
5,6,1,David Sterritt,Although it s less novel and feisty than the o...,75.0,expert
6,7,1,Susan Wloszczyna,NaN,88.0,expert
7,8,1,Paula Nechak,Lacks the cohesive flow of Fantasia and suffer...,75.0,expert
8,9,1,Jay Carr,NaN,75.0,expert
9,10,1,Lawrence Toppman,State-of-the-art.,75.0,expert
